# Logic Lab — Logical Planning

## What this notebook solves

**Goal:** build and test a breadth-first logical planner that transports one package from location A to C. The lab's central idea is: **Logic + Search = Planning**.

Source task: `logic_lab_ex.pdf`  
Original program converted here: `warehouse_planner.py`


> **How to use this notebook.** Read the task sections in order, then run the implementation cell and the experiment cells. The original Python source is preserved below as executable notebook code; it has not been rewritten into a different algorithm.

The task wording and requirements below are transcribed and condensed from the lab PDF in this folder. The completed answers, prompt, test evidence, and reflections follow `logic_lab_solution.md`.


## Task 0 — Understand the planning problem

### Question from the PDF

Specify the initial state `I`, goal `G`, available actions, and every action's preconditions and effects. Then decide whether `PickUp(Package, A)` and `Drop(Package, C)` are initially applicable.

### Solution

| Component | Specification |
| --- | --- |
| Initial state `I` | `{At(Robot, A), At(Package, A)}` |
| Goal `G` | `{At(Package, C)}` |
| Move actions | `Move(A,B)`, `Move(B,A)`, `Move(B,C)`, `Move(C,B)` |
| Pick-up actions | `PickUp(Package, X)` at `X ∈ {A,B,C}` |
| Drop actions | `Drop(Package, X)` at `X ∈ {A,B,C}` |

`Move(X,Y)` requires `At(Robot, X)` and replaces it with `At(Robot, Y)`. `PickUp(Package, X)` requires robot and package at `X` and not already holding it; it adds `Holding(Package)` and removes `At(Package, X)`. `Drop(Package, X)` requires robot at `X` and `Holding(Package)`; it adds `At(Package, X)` and removes `Holding(Package)`.

Initially, `PickUp(Package, A)` **is applicable**: both required location facts are present and the robot is not holding the package. `Drop(Package, C)` **is not applicable**: neither `At(Robot, C)` nor `Holding(Package)` is true.


## Task 1 — Construct a plan by hand

### Question from the PDF

Find a valid plan to move the package from A to C and record the state after every action.

### Worked plan

| State | Action just taken | Facts after the action |
| --- | --- | --- |
| `S0` | initial | `At(Robot,A)`, `At(Package,A)` |
| `S1` | `PickUp(Package,A)` | `At(Robot,A)`, `Holding(Package)` |
| `S2` | `Move(A,B)` | `At(Robot,B)`, `Holding(Package)` |
| `S3` | `Move(B,C)` | `At(Robot,C)`, `Holding(Package)` |
| `S4` | `Drop(Package,C)` | `At(Robot,C)`, `At(Package,C)` |

`S4` satisfies the goal because it contains `At(Package,C)`. The PDF's example mentioning `PickUp(Package,B)` is not applicable from the supplied initial state: the package starts at A.


## Task 2 — Implement the planner

### Question from the PDF

Represent states as sets of propositions; model actions with names, positive/negative preconditions and effects; use BFS; report no plan when appropriate; print actions and reached states.

### Design map

- **Preconditions:** `Action.applicable` checks that positive facts are included and negative facts are absent.
- **Effects:** `Action.apply` removes negative effects, then adds positive effects.
- **Goal test:** `target <= state` allows a goal to be a subset of a richer state.
- **Search:** `bfs_plan` explores applicable successor states in breadth-first order and remembers visited states.

The next cell is the complete converted implementation. Run it to execute its three required tests.


In [2]:
"""Breadth-first logical planner for the warehouse-robot laboratory."""

from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from typing import Iterable, Optional


@dataclass(frozen=True)
class Action:
    name: str
    positive_preconditions: frozenset[str] = frozenset()
    negative_preconditions: frozenset[str] = frozenset()
    positive_effects: frozenset[str] = frozenset()
    negative_effects: frozenset[str] = frozenset()

    def applicable(self, state: frozenset[str]) -> bool:
        return (self.positive_preconditions <= state and
                self.negative_preconditions.isdisjoint(state))

    def apply(self, state: frozenset[str]) -> frozenset[str]:
        return (state - self.negative_effects) | self.positive_effects


def bfs_plan(initial_state: Iterable[str], actions: Iterable[Action], goal: Iterable[str]) -> Optional[list[tuple[Action, frozenset[str]]]]:
    """Return a shortest plan and the state after every action, or None."""
    initial, target, actions = frozenset(initial_state), frozenset(goal), tuple(actions)
    if target <= initial:
        return []
    frontier, visited = deque([(initial, [])]), {initial}
    while frontier:
        state, path = frontier.popleft()
        for action in actions:
            if not action.applicable(state):
                continue
            successor = action.apply(state)
            if successor in visited:
                continue
            next_path = path + [(action, successor)]
            if target <= successor:
                return next_path
            visited.add(successor)
            frontier.append((successor, next_path))
    return None


def warehouse_actions(include_pickup: bool = True) -> list[Action]:
    actions: list[Action] = []
    for source, destination in (("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")):
        actions.append(Action(f"Move({source}, {destination})", frozenset({f"At(Robot, {source})"}),
                              positive_effects=frozenset({f"At(Robot, {destination})"}),
                              negative_effects=frozenset({f"At(Robot, {source})"})))
    for location in ("A", "B", "C"):
        if include_pickup:
            actions.append(Action(f"PickUp(Package, {location})",
                                  frozenset({f"At(Robot, {location})", f"At(Package, {location})"}),
                                  frozenset({"Holding(Package)"}),
                                  frozenset({"Holding(Package)"}),
                                  frozenset({f"At(Package, {location})"})))
        actions.append(Action(f"Drop(Package, {location})",
                              frozenset({f"At(Robot, {location})", "Holding(Package)"}),
                              positive_effects=frozenset({f"At(Package, {location})"}),
                              negative_effects=frozenset({"Holding(Package)"})))
    return actions


def show_test(name: str, initial: set[str], actions: list[Action], goal: set[str]) -> None:
    print(f"\n{name}\nInitial: {sorted(initial)}\nGoal:    {sorted(goal)}")
    plan = bfs_plan(initial, actions, goal)
    if plan is None:
        print("No plan found")
        return
    print("Plan found:")
    state = frozenset(initial)
    for number, (action, reached) in enumerate(plan, 1):
        assert action.applicable(state), f"invalid action: {action.name}"
        assert action.apply(state) == reached, f"wrong transition: {action.name}"
        print(f"  {number}. {action.name} -> {sorted(reached)}")
        state = reached
    assert frozenset(goal) <= state, "plan did not reach the goal"
    print("Plan is valid: goal reached.")


if __name__ == "__main__":
    initial = {"At(Robot, A)", "At(Package, A)"}
    goal = {"At(Package, C)"}
    show_test("Test A - solvable problem", initial, warehouse_actions(), goal)
    show_test("Test B - impossible problem", initial, warehouse_actions(False), goal)
    robot_only_moves = [
        Action("Move(A, B)", frozenset({"At(Robot, A)"}), positive_effects=frozenset({"At(Robot, B)"}), negative_effects=frozenset({"At(Robot, A)"})),
        Action("Move(B, C)", frozenset({"At(Robot, B)"}), positive_effects=frozenset({"At(Robot, C)"}), negative_effects=frozenset({"At(Robot, B)"})),
    ]
    show_test("Test C - irrelevant robot-only actions", initial, robot_only_moves, goal)



Test A - solvable problem
Initial: ['At(Package, A)', 'At(Robot, A)']
Goal:    ['At(Package, C)']
Plan found:
  1. PickUp(Package, A) -> ['At(Robot, A)', 'Holding(Package)']
  2. Move(A, B) -> ['At(Robot, B)', 'Holding(Package)']
  3. Move(B, C) -> ['At(Robot, C)', 'Holding(Package)']
  4. Drop(Package, C) -> ['At(Package, C)', 'At(Robot, C)']
Plan is valid: goal reached.

Test B - impossible problem
Initial: ['At(Package, A)', 'At(Robot, A)']
Goal:    ['At(Package, C)']
No plan found

Test C - irrelevant robot-only actions
Initial: ['At(Package, A)', 'At(Robot, A)']
Goal:    ['At(Package, C)']
No plan found


## Task 3 — Test the generated planner

### Question from the PDF

Run: (A) the solvable warehouse, (B) an impossible version without `PickUp`, and (C) a case with irrelevant robot-only moves. Record initial state, goal, whether a plan is found, plan, and validity.

### Expected results from the converted program

| Test | Plan found? | Result / validity |
| --- | --- | --- |
| A — solvable | Yes | `PickUp(A) → Move(A,B) → Move(B,C) → Drop(C)`; every transition is checked and the goal is reached. |
| B — no pick-up action | No | `No plan found`; the package cannot leave A. |
| C — robot-only moves | No | `No plan found`; reaching C with only the robot does not satisfy `At(Package,C)`. |

The `assert` statements in `show_test` independently check each action's applicability, its exact successor state, and the final goal.


In [3]:
# Task 3: executable tests for the three scenarios required in the lab sheet
test_initial = {"At(Robot, A)", "At(Package, A)"}
test_goal = {"At(Package, C)"}

# Test A — original warehouse: a valid plan must be found.
solvable_plan = bfs_plan(test_initial, warehouse_actions(), test_goal)
assert solvable_plan is not None
assert [action.name for action, _ in solvable_plan] == [
    "PickUp(Package, A)", "Move(A, B)", "Move(B, C)", "Drop(Package, C)"
]
assert test_goal <= solvable_plan[-1][1]

# Test B — no PickUp actions: the package cannot leave A.
impossible_plan = bfs_plan(test_initial, warehouse_actions(include_pickup=False), test_goal)
assert impossible_plan is None

# Test C — robot-only movement does not achieve a package-delivery goal.
robot_only_actions = [
    Action("Move(A, B)", frozenset({"At(Robot, A)"}),
           positive_effects=frozenset({"At(Robot, B)"}),
           negative_effects=frozenset({"At(Robot, A)"})),
    Action("Move(B, C)", frozenset({"At(Robot, B)"}),
           positive_effects=frozenset({"At(Robot, C)"}),
           negative_effects=frozenset({"At(Robot, B)"})),
]
irrelevant_plan = bfs_plan(test_initial, robot_only_actions, test_goal)
assert irrelevant_plan is None

print("Test A passed: valid delivery plan found.")
print("Test B passed: no plan found without PickUp.")
print("Test C passed: robot-only actions do not satisfy the package goal.")


Test A passed: valid delivery plan found.
Test B passed: no plan found without PickUp.
Test C passed: robot-only actions do not satisfy the package goal.


## Task 4 — Logic and search

### Complete the PDF flow

`Current state → Check action preconditions → Action applicable? → Generate successor state → Search over alternatives → Goal?`

Logic answers *which actions are legal* and how facts change. BFS answers *which legal action sequence to try next*. Keeping these responsibilities separate prevents an action merely listed in the domain from being used illegally.

## Task 5 — Verify an LLM plan

The LLM-generated plan is verified below against the planner's declared preconditions and effects. The proposed plan is `PickUp(Package,A) → Move(A,B) → Move(B,C) → Drop(Package,C)`.

| Step | Current state | Action and required preconditions | Verification | State after applying effects |
| --- | --- | --- | --- | --- |
| 0 | `At(Robot,A)`, `At(Package,A)` | `PickUp(Package,A)` requires `At(Robot,A)`, `At(Package,A)`, and absence of `Holding(Package)` | All conditions hold. | `At(Robot,A)`, `Holding(Package)` |
| 1 | `At(Robot,A)`, `Holding(Package)` | `Move(A,B)` requires `At(Robot,A)` | The required fact holds. | `At(Robot,B)`, `Holding(Package)` |
| 2 | `At(Robot,B)`, `Holding(Package)` | `Move(B,C)` requires `At(Robot,B)` | The required fact holds. | `At(Robot,C)`, `Holding(Package)` |
| 3 | `At(Robot,C)`, `Holding(Package)` | `Drop(Package,C)` requires `At(Robot,C)` and `Holding(Package)` | Both conditions hold. | `At(Robot,C)`, `At(Package,C)` |

The final state contains `At(Package,C)`, so it satisfies the goal. The program independently performs this same verification: `Action.applicable` checks preconditions, `Action.apply` computes the successor state, and the `assert` statements confirm each transition and the final goal.

The independently executed state transitions should be trusted more than the LLM explanation. An explanation can sound plausible while missing a required fact, but the program checks the declared rules against each actual state in a repeatable way.


## Tasks 6–8 — Optional Prolog verifier

### Task 6 — Prolog as a plan verifier

Create a file named `planner.pl` using the program in the next cell. In SWI-Prolog, load it from the folder containing the file with `?- [planner].` The expected response is `true.` when the file loads successfully.

#### Queries to run and expected output

| Prolog query | Desired output | Why |
| --- | --- | --- |
| `?- can_move(a,b).` | `true.` | `connected(a,b).` is a warehouse fact, so the rule derives `can_move(a,b)`. |
| `?- can_move(a,c).` | `false.` | There is no direct fact `connected(a,c).`; the rule cannot derive the query. |

The rule `can_move(X,Y) :- connected(X,Y).` is the executable Prolog form of the implication `Connected(X,Y) → CanMove(X,Y)`. A two-step route from `a` to `c` does not make `can_move(a,c)` true, because this rule checks for a direct connection only.

### Task 7

`valid_move(a,b)` and `valid_move(b,c)` succeed; `valid_move(a,c)` fails. Therefore a proposed direct `Move(a,c)` is unsupported. Prolog serves as an independent logical check of a candidate action.

### Task 8

For `wet_road.`, `slippery :- wet_road.`, and `reduce_speed :- slippery.`, the reasoning is: `wet_road ⇒ slippery ⇒ reduce_speed`; hence the query succeeds.

The query succeeds because `wet_road` is a fact, which derives `slippery`, which in turn derives `reduce_speed`.


In [4]:
# Copy the text below into a file named planner.pl, then load it in SWI-Prolog.
planner_pl = """% Warehouse connection facts (Task 6)
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

% Task 6: a move is possible when the locations are directly connected.
can_move(X,Y) :-
    connected(X,Y).

% Task 7: independently verify a proposed move.
valid_move(X,Y) :-
    connected(X,Y).

% Task 8: facts and rules for an implication chain.
wet_road.
slippery :-
    wet_road.
reduce_speed :-
    slippery.
"""

print(planner_pl)


% Warehouse connection facts (Task 6)
connected(a,b).
connected(b,a).
connected(b,c).
connected(c,b).

% Task 6: a move is possible when the locations are directly connected.
can_move(X,Y) :-
    connected(X,Y).

% Task 7: independently verify a proposed move.
valid_move(X,Y) :-
    connected(X,Y).

% Task 8: facts and rules for an implication chain.
wet_road.
slippery :-
    wet_road.
reduce_speed :-
    slippery.



## Implementation, test notes, and reflection

**Prompt used:**

> Implement a Python planning agent. A state is a set of logical propositions. Each action has a name, positive/negative preconditions, and positive/negative effects. An action is applicable only if all positive preconditions are in the state and negative preconditions are absent. Apply it by deleting negative effects then adding positive effects. Use BFS to return a plan to a positive goal, or no plan if impossible. Print the plan and reached state after each action. Use the A-B-C warehouse problem.

`applicable` performs the precondition test, `apply` performs the effects, `target <= state` is the goal test, and `bfs_plan` uses a FIFO queue with visited states. BFS therefore returns a shortest plan when all actions have equal cost.

All Task 3 tests start with `I = {At(Robot,A), At(Package,A)}` and use `G = {At(Package,C)}`. Test A finds the valid delivery plan; Test B reports no plan when `PickUp` actions are removed; and Test C reports no plan for robot-only movement.

Explicit preconditions and effects make the world model unambiguous. They prevent illegal actions such as dropping a package without holding it. A plausible-looking plan still needs verification because a required fact may be absent. The LLM contributed the action and BFS implementation, while every transition, test result, and final goal was independently checked. Planning is state-space search: states are nodes, applicable actions are edges, and the goal test determines success.

Independently executed state transitions should be trusted more than an LLM explanation, because executing declared rules against actual states is repeatable verification.
